In [2]:
# loading and setting up
import pandas as pd
from IPython.display import display
pd.set_option('display.max_rows',100)
mission_launches = pd.read_csv('mission_launches.csv')
display(mission_launches.head())

,Unnamed: 0.1,Unnamed: 0,Organisation,Location,Date,Detail,Rocket_Status,Price,Mission_Status
0,0,0,SpaceX,"LC-39A, Kennedy Space Center, Florida, USA","Fri Aug 07, 2020 05:12 UTC",Falcon 9 Block 5 | Starlink V1 L9 & BlackSky,StatusActive,50.0,Success
1,1,1,CASC,"Site 9401 (SLS-2), Jiuquan Satellite Launch Ce...","Thu Aug 06, 2020 04:01 UTC",Long March 2D | Gaofen-9 04 & Q-SAT,StatusActive,29.75,Success
2,2,2,SpaceX,"Pad A, Boca Chica, Texas, USA","Tue Aug 04, 2020 23:57 UTC",Starship Prototype | 150 Meter Hop,StatusActive,NaN,Success
3,3,3,Roscosmos,"Site 200/39, Baikonur Cosmodrome, Kazakhstan","Thu Jul 30, 2020 21:25 UTC",Proton-M/Briz-M | Ekspress-80 & Ekspress-103,StatusActive,65.0,Success
4,4,4,ULA,"SLC-41, Cape Canaveral AFS, Florida, USA","Thu Jul 30, 2020 11:50 UTC",Atlas V 541 | Perseverance,StatusActive,145.0,Success


In [3]:
# cleanup
mission_launches.drop(mission_launches.columns[[0, 1]], axis=1, inplace=True)
display(mission_launches.head())

,Organisation,Location,Date,Detail,Rocket_Status,Price,Mission_Status
0,SpaceX,"LC-39A, Kennedy Space Center, Florida, USA","Fri Aug 07, 2020 05:12 UTC",Falcon 9 Block 5 | Starlink V1 L9 & BlackSky,StatusActive,50.0,Success
1,CASC,"Site 9401 (SLS-2), Jiuquan Satellite Launch Ce...","Thu Aug 06, 2020 04:01 UTC",Long March 2D | Gaofen-9 04 & Q-SAT,StatusActive,29.75,Success
2,SpaceX,"Pad A, Boca Chica, Texas, USA","Tue Aug 04, 2020 23:57 UTC",Starship Prototype | 150 Meter Hop,StatusActive,NaN,Success
3,Roscosmos,"Site 200/39, Baikonur Cosmodrome, Kazakhstan","Thu Jul 30, 2020 21:25 UTC",Proton-M/Briz-M | Ekspress-80 & Ekspress-103,StatusActive,65.0,Success
4,ULA,"SLC-41, Cape Canaveral AFS, Florida, USA","Thu Jul 30, 2020 11:50 UTC",Atlas V 541 | Perseverance,StatusActive,145.0,Success


In [4]:
#adding missing column
mission_launches['Date'] = pd.to_datetime(mission_launches['Date'] , errors='coerce')
mission_launches = mission_launches[mission_launches['Date'].notna()]
mission_launches['Year'] = mission_launches['Date'].dt.year.astype('Int64')
mission_launches['Month'] = mission_launches['Date'].dt.month.astype('Int64')
display(mission_launches.head())

,Organisation,Location,Date,Detail,Rocket_Status,Price,Mission_Status,Year,Month
0,SpaceX,"LC-39A, Kennedy Space Center, Florida, USA",2020-08-07 05:12:00+00:00,Falcon 9 Block 5 | Starlink V1 L9 & BlackSky,StatusActive,50.0,Success,2020,8
1,CASC,"Site 9401 (SLS-2), Jiuquan Satellite Launch Ce...",2020-08-06 04:01:00+00:00,Long March 2D | Gaofen-9 04 & Q-SAT,StatusActive,29.75,Success,2020,8
2,SpaceX,"Pad A, Boca Chica, Texas, USA",2020-08-04 23:57:00+00:00,Starship Prototype | 150 Meter Hop,StatusActive,NaN,Success,2020,8
3,Roscosmos,"Site 200/39, Baikonur Cosmodrome, Kazakhstan",2020-07-30 21:25:00+00:00,Proton-M/Briz-M | Ekspress-80 & Ekspress-103,StatusActive,65.0,Success,2020,7
4,ULA,"SLC-41, Cape Canaveral AFS, Florida, USA",2020-07-30 11:50:00+00:00,Atlas V 541 | Perseverance,StatusActive,145.0,Success,2020,7


In [5]:
#mission per year / organization
mission_per_year_per_organization = mission_launches.groupby(['Year' , 'Organisation']).agg(launch_count=('Organisation', 'size')).reset_index().sort_values(by='launch_count', ascending=False)
display(mission_per_year_per_organization.head())

,Year,Organisation,launch_count
126,1976,RVSN USSR,93
133,1977,RVSN USSR,92
87,1971,RVSN USSR,90
119,1975,RVSN USSR,88
74,1970,RVSN USSR,86


In [6]:
#launches per month
mission_per_year_per_organization = mission_launches.groupby(['Month']).agg(launch_count=('Month', 'size')).reset_index().sort_values(by='launch_count', ascending=False)
display(mission_per_year_per_organization.head())

,Month,launch_count
11,12,430
5,6,386
9,10,375
3,4,366
7,8,358


In [7]:
#lauch cost
mission_launches['Price'] = pd.to_numeric(mission_launches['Price'] , errors='coerce')
mission_launches = mission_launches[mission_launches['Price'].notna()]
cost_per_year = mission_launches.groupby(['Year']).agg(cost=('Price', 'sum') , nbr_mission=('Price' , 'size')).reset_index().sort_values(by='Year', ascending=True)
cost_per_year['cost_per_launch'] = cost_per_year['cost'] / cost_per_year['nbr_mission']
display(cost_per_year.head())

,Year,cost,nbr_mission,cost_per_launch
0,1964,126.46,2,63.23
1,1965,126.46,2,63.23
2,1966,177.00,3,59.00
3,1967,354.00,6,59.00
4,1968,472.00,8,59.00


In [8]:
#mission safety
safety_stats = pd.crosstab(mission_launches['Year'], mission_launches['Mission_Status']).sort_values(by='Year', ascending=False)
safety_stats['Total'] = safety_stats.sum(axis=1)
display(safety_stats.head())


Mission_Status,Failure,Partial Failure,Prelaunch Failure,Success,Total
Year,,,,,
2020,4,0,0,48,52
2019,2,0,0,71,73
2018,0,1,0,87,88
2017,2,2,0,62,66
2016,1,1,1,61,64


In [ ]:
%%sql
